# Advanced Retrieval Pipeline with Metadata Filtering and Query Refinement

This notebook implements a retrieval system that:
1.  Analyzes user queries to extract metadata filters (Year, Dept, etc.).
2.  Refines the query for better semantic matching.
3.  Performs a filtered similarity search in ChromaDB.

In [1]:
import os
import json
from typing import List, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv("../.env")

# Configuration
DB_DIR = "../metadata_n_db/chroma_db"

if not os.path.exists(DB_DIR):
    print(f"Warning: DB directory {DB_DIR} does not exist. Please check the path.")
else:
    print(f"DB Directory: {os.path.abspath(DB_DIR)}")

DB Directory: /home/rishabh/coding/pro/RAG/metadata_n_db/chroma_db


In [2]:
class RAGRetriever:
    def __init__(self, db_dir: str, api_key_env: str = "GEMINI1", model_name: str = "gemini-1.5-flash"):
        """
        Initialize the RAG Retriever with Embeddings, Vector Store, BM25, and LLM.
        """
        self.api_key = os.getenv(api_key_env)
        if not self.api_key:
            raise ValueError(f"API Key environment variable '{api_key_env}' not found.")
            
        print(f"Initializing RAGRetriever with model: {model_name}")
        
        # 1. Initialize LLM
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0,
            google_api_key=self.api_key
        )
        
        # 2. Initialize Embeddings
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            google_api_key=self.api_key
        )
        
        # 3. Load Vector Store
        self.vectorstore = Chroma(
            persist_directory=db_dir,
            embedding_function=self.embeddings
        )
        print(f"Vector Store loaded with {self.vectorstore._collection.count()} documents.")
        
        # 4. Initialize BM25 (Keyword Search)
        print("Building BM25 Index from Vector Store documents...")
        self._build_bm25_index()
        
        # 5. Setup Chains
        self._setup_reranking_chain()
        self._setup_multi_query_chain()
        
    def _build_bm25_index(self):
        """
        Fetches all documents from ChromaDB to build an in-memory BM25 index.
        """
        try:
            data = self.vectorstore.get() 
            docs = []
            if data and data['documents']:
                for i, text in enumerate(data['documents']):
                    metadata = data['metadatas'][i] if data['metadatas'] else {}
                    docs.append(Document(page_content=text, metadata=metadata))
            
            if not docs:
                print("Warning: No documents found in Vector Store to build BM25 index.")
                self.bm25_retriever = None
                return

            self.bm25_retriever = BM25Retriever.from_documents(docs)
            print(f"BM25 Index built with {len(docs)} documents.")
            
        except Exception as e:
            print(f"Error building BM25 index: {e}")
            self.bm25_retriever = None

    def _setup_reranking_chain(self):
        """Sets up the LLM chain used for reranking documents."""
        class RelevanceScore(BaseModel):
            index: int = Field(description="The index of the document in the provided list")
            relevance_score: float = Field(description="A score from 0.0 to 1.0 indicating relevance")
            reasoning: str = Field(description="Brief reason why this document matches the constraints")

        class RankedDocuments(BaseModel):
            ranked_results: List[RelevanceScore]

        self.rerank_parser = JsonOutputParser(pydantic_object=RankedDocuments)

        self.rerank_prompt = PromptTemplate(
            template="""You are an expert relevance ranker. 
            The user asked: "{query}"
            
            Below is a list of document snippets retrieved from a database. 
            Your job is to evaluate each snippet and determine if it truly answers the user's specific constraints (e.g., specific year, specific department, specific format).
            
            If a document is relevant, assign a high score (0.7 - 1.0).
            If it is topic-adjacent but misses the specific constraint (e.g., wrong year), assign a low score (0.0 - 0.3).
            
            Documents:
            {doc_list}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["query", "doc_list"],
            partial_variables={"format_instructions": self.rerank_parser.get_format_instructions()},
        )

        self.rerank_chain = self.rerank_prompt | self.llm | self.rerank_parser

    def _setup_multi_query_chain(self):
        """Sets up the LLM chain for generating multiple query variations."""
        class MultiQuery(BaseModel):
            queries: List[str] = Field(description="List of 3 alternative versions of the user query")

        self.mq_parser = JsonOutputParser(pydantic_object=MultiQuery)

        self.mq_prompt = PromptTemplate(
            template="""You are an AI assistant. Your task is to generate 3 different versions of the given user question to retrieve relevant documents from a vector database.Use different wording which is more formal.
            By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
            
            Original Question: {question}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["question"],
            partial_variables={"format_instructions": self.mq_parser.get_format_instructions()},
        )

        self.mq_chain = self.mq_prompt | self.llm | self.mq_parser

    def generate_queries(self, original_query: str) -> List[str]:
        """Generates 3 variations of the query using the LLM."""
        try:
            result = self.mq_chain.invoke({"question": original_query})
            queries = result.get("queries", [])
            # Ensure original query is included
            if original_query not in queries:
                queries.insert(0, original_query)
            return queries[:4] # Return max 4 queries (original + 3 generated)
        except Exception as e:
            print(f"Multi-query generation failed: {e}")
            return [original_query]

    def reciprocal_rank_fusion(self, results: List[List[Document]], k=60):
        """
        Combines multiple lists of ranked documents using Reciprocal Rank Fusion (RRF).
        """
        fused_scores = {}
        doc_map = {} 

        for rank_list in results:
            for rank, doc in enumerate(rank_list):
                doc_key = doc.page_content
                if doc_key not in fused_scores:
                    fused_scores[doc_key] = 0
                    doc_map[doc_key] = doc
                fused_scores[doc_key] += 1 / (rank + k)

        reranked_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_map[doc_str] for doc_str, score in reranked_results]

    def retrieve(self, query: str, k_fetch: int = 10, top_n: int = 3) -> List:
        """
        Retrieves documents using Multi-Query + Hybrid Search + RRF + LLM Reranking.
        """
        print(f"--- 1. Multi-Query Generation for: '{query}' ---")
        queries = self.generate_queries(query)
        print(f"Generated Queries: {queries}")
        
        all_results_lists = []
        
        print(f"\n--- 2. Hybrid Retrieval for each query ---")
        for q in queries:
            # Vector Search
            vector_docs = self.vectorstore.similarity_search(q, k=k_fetch)
            all_results_lists.append(vector_docs)
            
            # Keyword Search
            if self.bm25_retriever:
                self.bm25_retriever.k = k_fetch
                keyword_docs = self.bm25_retriever.invoke(q)
                all_results_lists.append(keyword_docs)
        
        # Fusion
        print(f"\n--- 3. RRF Fusion of {len(all_results_lists)} result lists ---")
        initial_docs = self.reciprocal_rank_fusion(all_results_lists, k=60)
        
        # Slice to keep context window reasonable
        initial_docs = initial_docs[:k_fetch * 2] # Allow slightly more docs for reranker
        
        print(f"[Log] Top {len(initial_docs)} Documents after Fusion:")
        for i, doc in enumerate(initial_docs[:5]): # Print top 5 only to avoid clutter
            print(f"  [{i}] Source: {doc.metadata.get('source')} | Title: {doc.metadata.get('title')}")
        
        # Format for LLM
        doc_texts = []
        for i, doc in enumerate(initial_docs):
            snippet = f"Doc ID {i}:\nMetadata: {doc.metadata}\nContent: {doc.page_content[:400]}..." 
            doc_texts.append(snippet)
        
        combined_text = "\n\n".join(doc_texts)

        print(f"\n--- 4. Reranking {len(initial_docs)} documents ---")
        try:
            ranking_result = self.rerank_chain.invoke({"query": query, "doc_list": combined_text})
            sorted_ranks = sorted(ranking_result['ranked_results'], key=lambda x: x['relevance_score'], reverse=True)
            
            final_docs = []
            print("\n--- Top Selected Documents ---")
            for item in sorted_ranks[:top_n]:
                if item['relevance_score'] < 0.5:
                    print(f"Skipping Doc {item['index']} (Low Score: {item['relevance_score']})")
                    continue
                    
                if 0 <= item['index'] < len(initial_docs):
                    original_doc = initial_docs[item['index']]
                    print(f"Score: {item['relevance_score']} | Doc Source: {original_doc.metadata.get('source')}")
                    print(f"Reasoning: {item['reasoning']}")
                    final_docs.append(original_doc)
                
            return final_docs

        except Exception as e:
            print(f"Reranking failed: {e}. Falling back to raw search results.")
            return initial_docs[:top_n]

In [3]:
# Initialize the Retriever
# You can switch models here (e.g., "gemini-2.5-flash-lite" if available)
retriever = RAGRetriever(
    db_dir=DB_DIR, 
    api_key_env="GEMINI2", 
    model_name="gemini-2.5-flash-lite"
)

Initializing RAGRetriever with model: gemini-2.5-flash-lite
Vector Store loaded with 2339 documents.
Building BM25 Index from Vector Store documents...
Vector Store loaded with 2339 documents.
Building BM25 Index from Vector Store documents...
BM25 Index built with 2339 documents.
BM25 Index built with 2339 documents.


In [4]:
# Test 2: Syllabus
retriever.retrieve("Syllabus of Mathematics-I for first year")

--- 1. Multi-Query Generation for: 'Syllabus of Mathematics-I for first year' ---
Generated Queries: ['Syllabus of Mathematics-I for first year', 'Curriculum outline for Mathematics I, first academic year', 'Academic syllabus for the introductory Mathematics I course, commencing in the initial year', 'Detailed course content for Mathematics I, targeting first-year students']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['Syllabus of Mathematics-I for first year', 'Curriculum outline for Mathematics I, first academic year', 'Academic syllabus for the introductory Mathematics I course, commencing in the initial year', 'Detailed course content for Mathematics I, targeting first-year students']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Title: Curricular Structure for B.Tech. I Year
  [1] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Title: 

[Document(id='6629a86c-ce04-4b99-86a9-8fc10173fb40', metadata={'author': 'Valued Customer', 'Dept': 'Institute', 'total_pages': 15, 'moddate': '2016-04-19T09:50:49+05:30', 'source': '../pdfs/1st_Year_Scheme_Syallbus.pdf', 'creator': 'Microsoft® Word 2013', 'page_label': '5', 'summary': 'This document outlines the curricular structure for the first year of the B.Tech. program, common to all branches at MNIT Jaipur.', 'title': 'Curricular Structure for B.Tech. I Year', 'doc_type': 'Syllabus', 'page': 4, 'audience': 'UG', 'creationdate': '2016-04-19T09:50:49+05:30', 'producer': 'Microsoft® Word 2013', 'year': 'Unknown'}, page_content='Differential Calculus :  Curvature , Concavity, convexity and points of  Inflexion, \nAsymptotes, Partial differentiation, Euler’s theorem on homogeneous functions, Total \ndifferentiation, Approximate calculation, Curve tracing (Cartesian and five polar curves - \nFolium of Descartes, Limacon, Cardioids, Lemniscates of Bernoulli and Equiangular \nspiral). \

In [6]:
# Test 3: Fee Structure (Specific Year)
retriever.retrieve("Fee Structure year 2016 admitted students for Btech students")

--- 1. Hybrid Retrieval for: 'Fee Structure year 2016 admitted students for Btech students' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 15 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/Final_Fee_Structure_PG_2017-18_admitted.pdf | Page: 1 | Title: Fee structure for M.Tech./M.Plan./MBA (Full-time) students
  [1] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [2] Source: ../pdfs/Fee_Structure_UG_2017-18.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [3] Source: ../pdfs/FeeUG2025-26.pdf | Page: 0 | Title: Fee Structure for B. Tech.
  [4] Source: ../pdfs/Fee_Structure_UG_2018-19_admitted.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [5] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 2 | Title: Fee Structure for B. Tech. /B.Arch.
  [6] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 1 | Title: Fee Structure for B. Tech. /B.Arch.
  [7] Source:

[Document(id='5cec31ff-209e-4db2-8492-8cd5b1a2ff75', metadata={'moddate': '2017-04-27T18:21:24+05:30', 'doc_type': 'Fee Structure', 'title': 'Fee Structure for B. Tech. /B.Arch.', 'creationdate': '2017-04-27T18:21:24+05:30', 'author': 'IBM', 'creator': 'Microsoft® Word 2013', 'source': '../pdfs/Fee_Structure_UG_2016-17.pdf', 'total_pages': 3, 'summary': 'This document details the fee structure for B.Tech and B.Arch students admitted in the 2016-17 session.', 'year': '2016', 'page': 0, 'audience': 'UG', 'Dept': 'Institute', 'producer': 'Microsoft® Word 2013', 'page_label': '1'}, page_content='MALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \n Fee Structure for B. Tech. /B.Arch. students admitted in the session 2016-17 \n \nTUITION FEE \n \nS. No. \nHead of Fee \n \nOdd Semester & Even Semester \nOP/OBC PH/SC/ST \nIncome  \nBelow 1 Lac \nIncome                                  \n1 Lac to 5 Lac \nIncome                   \nAbove 5 Lac All \n1. Tuition Fee per Semester 0 20,834.00 6

In [ ]:
# Test 4: Faculty Query
retriever.retrieve("Quantum computing classes for faculty")

--- 1. Wide Retrieval for: 'Quantum computing classes for faculty' ---
--- 2. Reranking 10 documents ---
--- 2. Reranking 10 documents ---

--- Top Selected Documents ---
Score: 1.0 | Doc Source: ../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Quantum Sensing' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-03_BASIC_QUANTUM_PROGRAMMING_2025.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Basic Quantum Programming' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-05_Quantum_Computation_Brochure_updated_1.pdf
Reasoning: This document is a notice for an 'Online Faculty P

[Document(id='168d713b-bef6-4d56-82f5-8998a0eda91a', metadata={'total_pages': 1, 'author': 'x', 'summary': 'An intensive 20-day online training programme on Quantum Sensing is being organized for faculty and doctoral students.', 'moddate': '2025-09-17T06:16:12+00:00', 'title': 'AICTE Approved Minor Course Curriculum on Quantum Computing', 'creator': 'Microsoft® Word 2016', 'doc_type': 'Notice', 'creationdate': '2025-09-17T06:16:12+00:00', 'page': 0, 'audience': 'Faculty', 'Dept': 'Institute', 'producer': 'www.ilovepdf.com', 'year': '2025', 'source': '../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf', 'page_label': '1'}, page_content='AICTE Approved Minor Course Curriculum \non Quantum Computing \n                  \n  \n \nIIT Kanpur, IIT Roorkee, IIT Guwahati,     \nNIT Patna, NIT Warangal, IIITDM Jabalpur \nOnline Faculty Programme on  \nQT – 07 :  \nQuantum Sensing   \nSept 26 – Oct 17, 2025 \nTwenty Days (Mon to Sat) \nTime: 2 – 4 PM (Daily 2 Hours) \n \nAn intensive 20-day-40-hour T

In [5]:
# Test 5: Policy
retriever.retrieve("What is the unfair means policy policy or UFM of the instituion")

--- 1. Multi-Query Generation for: 'What is the unfair means policy policy or UFM of the instituion' ---
Generated Queries: ['What is the unfair means policy policy or UFM of the instituion', "Could you please provide information regarding the institution's policy on unfair means or UFM?", 'I am seeking clarification on the established policy concerning unfair means (UFM) within the institution.', 'What are the institutional guidelines and regulations pertaining to unfair means (UFM)?']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['What is the unfair means policy policy or UFM of the instituion', "Could you please provide information regarding the institution's policy on unfair means or UFM?", 'I am seeking clarification on the established policy concerning unfair means (UFM) within the institution.', 'What are the institutional guidelines and regulations pertaining to unfair means (UFM)?']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result li

[Document(id='eb6ab811-f286-4ee2-b0b7-53961f48adc2', metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'moddate': '2015-04-23T17:12:10+05:30', 'audience': 'UG', 'doc_type': 'Policy', 'page': 4, 'year': 'Unknown', 'author': 'lavab', 'summary': 'This document outlines the policy regarding minimum academic performance requirements for B.Tech./B.Arch. students to progress to subsequent semesters at MNIT Jaipur.', 'title': 'Minimum Requirement to continue in the program & Promotion', 'Dept': 'Institute', 'creationdate': '2015-04-23T17:12:10+05:30', 'total_pages': 9, 'source': '../pdfs/Sem_promotion_policy_1st_year.pdf', 'page_label': '5'}, page_content='Students who have failed in one semester / taken semester withdrawal / rusticated fo r one \nsemester / not promoted to higher semester on account of N -4 rule or any other reason \n/Academically deficient student  not able to register for higher semester courses due to \nregistration of pending  

In [6]:
# Test 5: Policy
retriever.retrieve("what are guidlines for phd exam in year 2025")

--- 1. Multi-Query Generation for: 'what are guidlines for phd exam in year 2025' ---
Generated Queries: ['what are guidlines for phd exam in year 2025', 'What are the regulations for doctoral examinations scheduled for the year 2025?', 'Could you provide the official directives concerning PhD examinations for the 2025 academic year?', 'I am seeking information on the established protocols for PhD examinations occurring in 2025.']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['what are guidlines for phd exam in year 2025', 'What are the regulations for doctoral examinations scheduled for the year 2025?', 'Could you provide the official directives concerning PhD examinations for the 2025 academic year?', 'I am seeking information on the established protocols for PhD examinations occurring in 2025.']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/Guidelines.pdf | Title: Ph.D

[Document(id='367df7d2-d7d6-44b5-a1ed-9b4710d0a514', metadata={'audience': 'UG', 'source': '../pdfs/Guidelines.pdf', 'producer': 'Microsoft® Word 2013', 'title': 'Ph.D. Entrance Exam Guidelines', 'total_pages': 1, 'summary': 'This document outlines the guidelines for the Ph.D. entrance exam for the Even Semester 2025-26, including the selection process and requirements for shortlisted candidates.', 'moddate': '2025-12-02T14:48:48+05:30', 'doc_type': 'Guidelines', 'page_label': '1', 'year': '2025-26', 'page': 0, 'author': 'wipro', 'Dept': 'Institute', 'creationdate': '2025-12-02T14:48:48+05:30', 'creator': 'Microsoft® Word 2013'}, page_content='Guidelines for Ph.D. Entrance Exam, EVEN Semester 2025-26 \n \nWritten exam and interview for Ph.D. entrance exam, Even Semester 2025-26 will be \nheld during 09th and 10th December 2025 at MNIT campus.  \n \nSelection process will comprise of two steps (i) Written test (ii) Interview of \nshortlisted candidates. The written test will comprise of

In [7]:
# Test 5: Policy
retriever.retrieve("what is the college's perspective on Right to Information")

--- 1. Multi-Query Generation for: 'what is the college's perspective on Right to Information' ---
Generated Queries: ["what is the college's perspective on Right to Information", 'What is the official stance of the academic institution regarding the Right to Information Act?', "Could you provide information on the university's viewpoint concerning access to information legislation?", "What is the educational establishment's position on the principles and implementation of the Right to Information?"]

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ["what is the college's perspective on Right to Information", 'What is the official stance of the academic institution regarding the Right to Information Act?', "Could you provide information on the university's viewpoint concerning access to information legislation?", "What is the educational establishment's position on the principles and implementation of the Right to Information?"]

--- 2. Hybrid Retrieval for each query ---

[Document(id='0b66e958-9e65-4747-a40e-a80111cb7b6c', metadata={'creator': 'PScript5.dll Version 5.2.2', 'summary': 'This is an application form for requesting information under the Right to Information Act, 2005 at MNIT Jaipur.', 'total_pages': 1, 'Dept': 'Institute', 'doc_type': 'Application Form', 'source': '../pdfs/RTI-APPLICATION-FORM.pdf', 'year': 'Unknown', 'creationdate': '2011-04-19T16:58:10+05:30', 'page': 0, 'moddate': '2011-04-19T16:58:10+05:30', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '1', 'author': 'cse', 'audience': 'General', 'title': 'RTI Application Form'}, page_content='APPLICATION FORM (RIGHT TO INFORMATION ACT, 2005)  \nTo  \nThe Central Public Information Officer(……………………………………) \nMNIT JAIPUR  \nJLN Marg Jaipur – 302017.   \n \n1. Name of the Applicant: ___________________________________________________________   \n2. Address with PIN: ________________________________________________________________ \n  ______________________________________

In [9]:
retriever.retrieve("who were the gold medalist in year 2021-22")


--- 1. Multi-Query Generation for: 'who were the gold medalist in year 2021-22' ---
Generated Queries: ['who were the gold medalist in year 2021-22', 'Please identify the gold medal winners for the 2021-2022 period.', 'Could you provide a list of individuals or teams who achieved gold medals during the 2021-2022 sporting season?', 'What were the names of the gold medal recipients in the year 2021-22?']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['who were the gold medalist in year 2021-22', 'Please identify the gold medal winners for the 2021-2022 period.', 'Could you provide a list of individuals or teams who achieved gold medals during the 2021-2022 sporting season?', 'What were the names of the gold medal recipients in the year 2021-22?']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/Gold_Medalist.pdf | Title: Director’s Gold Medal
  [1] Source: ../pdfs/Gold_Medalist

[Document(id='b71cd549-0c1e-46b9-a3c5-7df7b3fa755a', metadata={'page': 0, 'author': 'IBM', 'producer': 'Microsoft® Word 2013', 'creationdate': '2023-03-31T16:08:35+05:30', 'year': '2021-22', 'audience': 'UG', 'source': '../pdfs/Gold_Medalist.pdf', 'Dept': 'Institute', 'summary': 'This document lists the recipients of the Director’s Gold Medal in B.Tech. and B.Arch. for the academic session 2021-22.', 'doc_type': 'Notice', 'page_label': '1', 'creator': 'Microsoft® Word 2013', 'moddate': '2023-03-31T16:08:35+05:30', 'title': 'Director’s Gold Medal', 'total_pages': 3}, page_content='1 \n \n \n \n \nMALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \nDirector’s Gold Medal in B. Tech. and B. Arch. in the academic session \n2021-22 \n \nS. \nNo ID No. Name Programme CGPA \n1 2017UAR1567 NUPUR MALIK ARCHITECTURE AND \nPLANNING 9.41 \n2 2018UCH1656 DARSHANA \nPALIWAL CHEMICAL ENGINEERING 9.66 \n3 2018UCE1103 ARSHIKA TOMAR CIVIL ENGINEERING 9.70 \n4 2018UCP1444 PRANSHU VYAS COMPUTER SCIENC